<a href="https://colab.research.google.com/github/NehaBommisetty07/AIBasedDentalDiagnosis/blob/main/DentalAI_RadiographSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ── Exact paths from your Google Drive ──────────────────────────
BASE     = '/content/drive/MyDrive/USE CASE - 01'
SEG_ROOT = Path(BASE) / 'Segmented Dental Radiography'
OPG_ROOT = Path(BASE) / 'Dental OPG images'
OUT_DIR  = Path(BASE) / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

# ── Verify ───────────────────────────────────────────────────────
print("BASE     :", BASE)
print("SEG_ROOT :", SEG_ROOT, "| Exists:", SEG_ROOT.exists())
print("OPG_ROOT :", OPG_ROOT, "| Exists:", OPG_ROOT.exists())

Mounted at /content/drive
BASE     : /content/drive/MyDrive/USE CASE - 01
SEG_ROOT : /content/drive/MyDrive/USE CASE - 01/Segmented Dental Radiography | Exists: True
OPG_ROOT : /content/drive/MyDrive/USE CASE - 01/Dental OPG images | Exists: True


In [ ]:
# Class exact names to check — run this first
import os
print("Folders inside Segmented Dental Radiography:")
for f in sorted(os.listdir('/content/drive/MyDrive/USE CASE - 01/Segmented Dental Radiography')):
    print("  →", repr(f))

Folders inside Segmented Dental Radiography:
  → '.DS_Store'
  → 'test'
  → 'train'
  → 'valid'


In [ ]:
print("Class folders inside train:")
for f in sorted(os.listdir('/content/drive/MyDrive/USE CASE - 01/Segmented Dental Radiography/train')):
    print("  →", repr(f))

Class folders inside train:
  → '.DS_Store'
  → 'Cavity'
  → 'Fillings'
  → 'Impacted Tooth'
  → 'Implant'
  → 'Normal'


In [ ]:
!pip install timm albumentations -q

import os, time, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, confusion_matrix, f1_score
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f' Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — Runtime → Change runtime type → T4 GPU')

 Device: cuda
GPU: Tesla T4


In [ ]:
# ── Exact class names from YOUR folder ──────────────────────
CLASS_NAMES  = ['Cavity', 'Fillings', 'Impacted Tooth', 'Implant', 'Normal']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES  = len(CLASS_NAMES)
IMG_SIZE     = 224
BATCH_SIZE   = 32
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8,8), p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
    A.RandomGamma(gamma_limit=(70,130), p=0.5),
    A.GaussNoise(var_limit=(5.0,30.0), p=0.4),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.12,
                       rotate_limit=12, border_mode=0, p=0.5),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])
val_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.CLAHE(clip_limit=2.0, p=1.0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

class DentalDataset(Dataset):
    def __init__(self, split, transform=None):
        self.transform = transform
        self.samples   = []
        root = SEG_ROOT / split
        for cls_name, idx in CLASS_TO_IDX.items():
            cls_path = root / cls_name
            if not cls_path.exists():
                print(f'  ⚠  Not found: {cls_path}')
                continue
            for ext in ['*.jpg','*.jpeg','*.JPG','*.png']:
                for p in cls_path.glob(ext):
                    self.samples.append((str(p), idx))
        counts = Counter(l for _,l in self.samples)
        print(f'  [{split:>5}] {len(self.samples):6,} images | '
              + '  '.join(f'{CLASS_NAMES[i]}:{counts[i]}' for i in range(NUM_CLASSES)))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((IMG_SIZE,IMG_SIZE), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        if self.transform: img = self.transform(image=img)['image']
        return img, label

    def weighted_sampler(self):
        counts = Counter(l for _,l in self.samples)
        n = len(self.samples)
        w = [n/(NUM_CLASSES*counts[l]) for _,l in self.samples]
        return WeightedRandomSampler(w, num_samples=n, replacement=True)

print('📦 Building datasets...')
train_ds = DentalDataset('train', train_tfm)
val_ds   = DentalDataset('valid', val_tfm)
test_ds  = DentalDataset('test',  val_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=train_ds.weighted_sampler(),
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'\nTrain batches: {len(train_loader)}')
print(f'   Val   batches: {len(val_loader)}')
print(f'   Test  batches: {len(test_loader)}')

📦 Building datasets...
  [train] 11,736 images | Cavity:0  Fillings:0  Impacted Tooth:0  Implant:0  Normal:11736
  [valid]    540 images | Cavity:0  Fillings:540  Impacted Tooth:0  Implant:0  Normal:0
  [ test]      0 images | Cavity:0  Fillings:0  Impacted Tooth:0  Implant:0  Normal:0

Train batches: 367
   Val   batches: 17
   Test  batches: 0


In [ ]:
# ── Exact class names from YOUR folder ──────────────────────
CLASS_NAMES  = ['Cavity', 'Fillings', 'Impacted Tooth', 'Implant', 'Normal']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES  = len(CLASS_NAMES)
IMG_SIZE     = 224
BATCH_SIZE   = 32
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8,8), p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
    A.RandomGamma(gamma_limit=(70,130), p=0.5),
    A.GaussNoise(var_limit=(5.0,30.0), p=0.4),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.12,
                       rotate_limit=12, border_mode=0, p=0.5),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])
val_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.CLAHE(clip_limit=2.0, p=1.0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

class DentalDataset(Dataset):
    def __init__(self, split, transform=None):
        self.transform = transform
        self.samples   = []
        root = SEG_ROOT / split
        for cls_name, idx in CLASS_TO_IDX.items():
            cls_path = root / cls_name
            if not cls_path.exists():
                print(f'  ⚠  Not found: {cls_path}')
                continue
            for ext in ['*.jpg','*.jpeg','*.JPG','*.png']:
                for p in cls_path.glob(ext):
                    self.samples.append((str(p), idx))
        counts = Counter(l for _,l in self.samples)
        print(f'  [{split:>5}] {len(self.samples):6,} images | '
              + '  '.join(f'{CLASS_NAMES[i]}:{counts[i]}' for i in range(NUM_CLASSES)))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((IMG_SIZE,IMG_SIZE), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        if self.transform: img = self.transform(image=img)['image']
        return img, label

    def weighted_sampler(self):
        counts = Counter(l for _,l in self.samples)
        n = len(self.samples)
        w = [n/(NUM_CLASSES*counts[l]) for _,l in self.samples]
        return WeightedRandomSampler(w, num_samples=n, replacement=True)

print('📦 Building datasets...')
train_ds = DentalDataset('train', train_tfm)
val_ds   = DentalDataset('valid', val_tfm)
test_ds  = DentalDataset('test',  val_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=train_ds.weighted_sampler(),
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'\n Train batches: {len(train_loader)}')
print(f'   Val   batches: {len(val_loader)}')
print(f'   Test  batches: {len(test_loader)}')


📦 Building datasets...
  [train] 11,736 images | Cavity:0  Fillings:0  Impacted Tooth:0  Implant:0  Normal:11736
  [valid]    540 images | Cavity:0  Fillings:540  Impacted Tooth:0  Implant:0  Normal:0
  [ test]      0 images | Cavity:0  Fillings:0  Impacted Tooth:0  Implant:0  Normal:0

 Train batches: 367
   Val   batches: 17
   Test  batches: 0


In [ ]:
class DentalClassifier(nn.Module):
    def __init__(self, num_classes=5, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b3', pretrained=True,
            num_classes=0, global_pool='avg'
        )
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.BatchNorm1d(512), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(512, 256),      nn.BatchNorm1d(256), nn.SiLU(), nn.Dropout(dropout/2),
            nn.Linear(256, num_classes)
        )
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
    def forward(self, x):
        return self.head(self.backbone(x))

model = DentalClassifier(num_classes=NUM_CLASSES).to(DEVICE)
model.freeze_backbone()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'EfficientNet-B3 loaded')
print(f'   Total params    : {total:,}')
print(f'   Trainable now   : {trainable:,}  (head only)')
print(f'   Classes         : {CLASS_NAMES}')

# Sanity check
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad(): out = model(dummy)
print(f'   Forward pass OK : {list(dummy.shape)} → {list(out.shape)}')

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

EfficientNet-B3 loaded
   Total params    : 11,617,325
   Trainable now   : 921,093  (head only)
   Classes         : ['Cavity', 'Fillings', 'Impacted Tooth', 'Implant', 'Normal']
   Forward pass OK : [2, 3, 224, 224] → [2, 5]


In [ ]:
dashboard_code = open('/content/drive/MyDrive/USE CASE - 01/dental_dashboard.py').read()
with open('dental_dashboard.py', 'w') as f:
    f.write(dashboard_code)
print("Dashboard file ready")

Dashboard file ready


In [ ]:
!pip install streamlit -q
print(" Streamlit installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 124.9 MB/s eta 0:00:00
 Streamlit installed


In [ ]:
import subprocess, time

subprocess.Popen(
    ["streamlit", "run", "dental_dashboard.py",
     "--server.port=8501", "--server.headless=true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print("="*50)
print("OPEN THIS URL:")
print(url)
print("="*50)
print("Open that URL → upload any OPG image → see full analysis!")

OPEN THIS URL:
https://8501-gpu-t4-s-kkb-usw4a1-15lu4on9lixww-a.us-west4-1.prod.colab.dev
Open that URL → upload any OPG image → see full analysis!


In [ ]:
# Check if streamlit is actually running
import subprocess
result = subprocess.run(['pgrep', '-f', 'streamlit'], capture_output=True, text=True)
print("Streamlit PID:", result.stdout.strip())

# Restart it cleanly
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
import time; time.sleep(2)

proc = subprocess.Popen(
    ["streamlit", "run", "dental_dashboard.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false",
     "--server.enableXsrfProtection=false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(6)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print("="*50)
print("OPEN THIS URL:")
print(url)
print("="*50)

Streamlit PID: 2295
OPEN THIS URL:
https://8501-gpu-t4-s-kkb-usw4a1-15lu4on9lixww-a.us-west4-1.prod.colab.dev
